# N7 — Reasoning em Agentes LMM

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a> 

Este notebook apresenta uma série de técnicas de raciocínio em agentes LLMs. O foco das técnicas está em **como o modelo chega à resposta de um problema** (tomada de decisão), e como podemos tornar esse raciocínio ``visível''.
 
Vamos estudar sete técnicas de reasoning amplamente discutidas na literatura de LLMs:

1. **Few-Shot Prompting** — orienta o modelo por exemplo, sem necessariamente expor raciocínio;
2. **Chain-of-Thought (CoT)** — pede ao modelo para raciocinar passo a passo antes de responder;
3. **Self-Ask** — o modelo decompõe a pergunta em subperguntas explícitas, respondendo cada uma;
4. **Self-Consistency** — gera várias cadeias de raciocínio independentes e escolhe a resposta mais consistente entre elas;
5. **ReAct** — intercala raciocínio e ação;
6. **Least-to-Most** — decompõe o problema do mais simples para o mais complexo, resolvendo em ordem;
7. **Tree of Thoughts (ToT)** — explora múltiplos caminhos de raciocínio em paralelo.

**Para as técnicas que produzem raciocínio explícito, vamos sempre exibir esse raciocínio para o usuário**, não só a resposta final — isso ajuda o usuário a auditar, depurar e confiar (ou desconfiar, com base em evidência) na resposta do agente.

## 0.1 Pré-requisitos

Para acompanhar este notebook, é esperado que você já tenha:

- usado saída estruturada com `with_structured_output` e Pydantic (notebook de LangGraph com LLMs);
- visto o padrão ReAct e tool calling (notebook de Ferramentas);
- implementado ReAct e Tree of Thoughts em LangGraph (notebook de Planning) — vamos recapitular, não reconstruir do zero;
- familiaridade básica com `TypedDict`, `Literal` e listas em Python.

## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Diferenciar reasoning **implícito** (a resposta vem sem justificativa visível) de reasoning **explícito** (o raciocínio é parte da saída).
2. Usar Few-Shot Prompting para orientar formato e estilo de resposta.
3. Implementar Chain-of-Thought com o raciocínio separado da resposta final, e exibir ambos.
4. Implementar Self-Ask, decompondo uma pergunta em subperguntas visíveis.
5. Implementar Self-Consistency, exibindo todas as cadeias de raciocínio amostradas e a votação entre elas.
6. Relacionar ReAct e Tree of Thoughts (do notebook de Planning) como técnicas que também produzem raciocínio explícito.
7. Implementar Least-to-Most, exibindo a decomposição e cada subresposta.
8. Comparar as sete técnicas em termos de custo, visibilidade do raciocínio e adequação a diferentes problemas.

## 0.3 Mapa do notebook

Neste notebook, vamos passar por:

1. Reasoning implícito vs. explícito.
2. Few-Shot Prompting.
3. Chain-of-Thought.
4. Self-Ask.
5. Self-Consistency.
6. ReAct (recapitulação).
7. Least-to-Most.
8. Tree of Thoughts (recapitulação).
9. Comparando as sete técnicas.
10. Boas práticas.
11. Exercícios de fixação.
12. Resumo da aula.
13. Referências.

## 0.4 Contexto usado nos exemplos

Vamos continuar com o mesmo curso de agentes com LLMs e LangGraph.

O foco é **raciocinar sobre um problema com dados já disponíveis em texto**. Isso deixa mais fácil comparar as técnicas de reasoning sem misturar com decisões de quais tools chamar.

O problema motivador será o mesmo ao longo de toda a aula:

> Bruno está no módulo A5. Os módulos restantes do curso são:
> A5 (4 aulas), A6 (5 aulas), A7 (4 aulas), A8 (3 aulas), A9 (6 aulas), A10 (5 aulas).
> Cada aula dura 50 minutos. Bruno tem 10 dias antes da prova final e consegue
> estudar, no máximo, 2 aulas por dia. Ele consegue terminar o conteúdo a tempo?

Esse problema tem uma resposta correta e verificável (dá para conferir com uma continha), o que é útil para comparar as técnicas: **27 aulas restantes, a 2 aulas por dia, precisam de 14 dias — Bruno não consegue terminar em 10 dias.**

Vamos ver como cada técnica de reasoning chega (ou não) a essa conclusão, e como cada uma expõe esse caminho para o usuário.

---

## 0.5 Preparação do ambiente

Este notebook assume que você já tem LangChain, LangGraph e um provedor de LLM configurados, como nos notebooks anteriores.

In [49]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
%pip install -U langchain langchain-core langgraph langchain-openai python-dotenv pydantic rich

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Agora importamos os elementos que serão usados ao longo da aula.

In [50]:
from __future__ import annotations

import json
import os
from collections import Counter
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.tools import tool as tool_decorator
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph

from IPython.display import Image, display
from rich import print
from rich.markdown import Markdown

In [52]:
def display_graph(graph: CompiledStateGraph) -> None:
    display(Image(graph.get_graph().draw_mermaid_png()))


def print_json(value: Any) -> None:
    """Imprime objetos Python como JSON formatado."""
    print(json.dumps(value, ensure_ascii=False, indent=2))


def print_reasoning(raciocinio: str, resposta_final: str) -> None:
    """Exibe o raciocínio explícito separado da resposta final.

    Usada em todas as técnicas deste notebook que produzem raciocínio visível,
    para manter um formato consistente de apresentação ao usuário.
    """
    print("[bold]Raciocínio:[/bold]")
    print(raciocinio)
    print("\n[bold]Resposta final:[/bold]")
    print(resposta_final)

Agora inicializamos o modelo e definimos o problema motivador, reutilizado em toda a aula.

In [54]:
load_dotenv()
print("Chave OPENAI_API_KEY configurada com sucesso." if "OPENAI_API_KEY" in os.environ else "Chave não encontrada.")

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

PROBLEMA_BRUNO = """Bruno está no módulo A5. Os módulos restantes do curso são:
A5 (4 aulas), A6 (5 aulas), A7 (4 aulas), A8 (3 aulas), A9 (6 aulas), A10 (5 aulas).
Cada aula dura 50 minutos. Bruno tem 10 dias antes da prova final e consegue
estudar, no máximo, 2 aulas por dia. Ele consegue terminar o conteúdo a tempo?"""

print(Markdown(PROBLEMA_BRUNO))

Chave OPENAI_API_KEY configurada com sucesso.

Bruno está no módulo A5. Os módulos restantes do curso são: A5 (4 aulas), A6 (5 aulas), A7 (4 aulas), A8 (3 aulas),
A9 (6 aulas), A10 (5 aulas). Cada aula dura 50 minutos. Bruno tem 10 dias antes da prova final e consegue estudar, 
no máximo, 2 aulas por dia. Ele consegue terminar o conteúdo a tempo?

# 1. Reasoning implícito vs. explícito

Antes de entrar em cada técnica, vale separar dois tipos de comportamento:

- **Reasoning implícito**: o modelo "pensa" internamente (na ativação da rede), mas a saída visível é só a resposta final — sem justificativa, sem passos, sem como chegou até ali.
- **Reasoning explícito**: o processo de raciocínio é **parte da saída** — passos numerados, subperguntas, ou um campo separado de "raciocínio" antes da resposta final.

Vamos ver isso na prática: pedir a resposta direta, sem nenhuma técnica.

In [55]:
resposta_direta = llm.invoke([
    SystemMessage(content="Responda apenas com 'sim' ou 'não', sem explicação."),
    HumanMessage(content=PROBLEMA_BRUNO),
])

print(resposta_direta.content)

Sim.

Esse tipo de resposta pode até estar certa, mas não é possível **auditar** como o modelo chegou até o resultado final, nem identificar em qual conta específica ele eventualmente errou. 

As técnicas a seguir mudam isso — a maioria delas expõe explicitamente os passos intermediários, e vamos sempre imprimir esse raciocínio para o usuário, não só a resposta final.

# 2. Few-Shot Prompting

## 2.1 Conceito

Few-Shot Prompting orienta o modelo por meio de **exemplos completos** de entrada e saída, dentro do próprio prompt, em vez de apenas descrever a tarefa. A ideia é que o modelo generalize o padrão dos exemplos para a nova pergunta.

Importante: few-shot, por si só, **não expõe raciocínio** — os exemplos podem ser apenas ``pergunta → resposta direta''. Se quisermos que o raciocínio apareça, os próprios exemplos precisam incluir o raciocínio (o que aproxima few-shot de Chain-of-Thought).

## 2.2 Few-shot sem raciocínio exposto

Vamos primeiro orientar apenas o **formato** da resposta, com exemplos de perguntas parecidas (mas mais simples) e suas respostas diretas.

In [61]:
EXEMPLOS_FEW_SHOT_SEM_RACIOCINIO = """
Pergunta: Uma aluna tem 6 aulas de 40 minutos para assistir em 3 dias, no máximo 2 por dia. Ela consegue?
Resposta: Sim.

Pergunta: Um aluno tem 10 aulas para assistir em 2 dias, no máximo 3 por dia. Ele consegue?
Resposta: Não.
"""

resposta_few_shot = llm.invoke([
    SystemMessage(content=f"""Responda apenas 'Sim.' ou 'Não.', seguindo o padrão dos exemplos abaixo.

{EXEMPLOS_FEW_SHOT_SEM_RACIOCINIO}"""),
    HumanMessage(content=f"Pergunta: {PROBLEMA_BRUNO}\nResposta:"),
])

print(resposta_few_shot.content)

Sim.

## 2.3 Few-shot com raciocínio exposto

Agora incluímos o raciocínio **dentro dos próprios exemplos**, para que o modelo também exponha o raciocínio na resposta nova.

In [57]:
EXEMPLOS_FEW_SHOT_COM_RACIOCINIO = """
Pergunta: Uma aluna tem 6 aulas de 40 minutos para assistir em 3 dias, no máximo 2 por dia. Ela consegue?
Raciocínio: Em 3 dias, a 2 aulas por dia, ela consegue assistir até 6 aulas. Ela precisa de exatamente 6. Portanto, consegue.
Resposta: Sim.

Pergunta: Um aluno tem 10 aulas para assistir em 2 dias, no máximo 3 por dia. Ele consegue?
Raciocínio: Em 2 dias, a 3 aulas por dia, ele consegue assistir até 6 aulas. Ele precisa de 10. 6 é menor que 10. Portanto, não consegue.
Resposta: Não.
"""

resposta_few_shot_cot = llm.invoke([
    SystemMessage(content=f"""Siga exatamente o padrão dos exemplos abaixo, incluindo um campo
'Raciocínio:' antes da 'Resposta:'.

{EXEMPLOS_FEW_SHOT_COM_RACIOCINIO}"""),
    HumanMessage(content=f"Pergunta: {PROBLEMA_BRUNO}"),
])

print(resposta_few_shot_cot.content)

Raciocínio: Bruno tem 4 aulas no módulo A5, 5 no A6, 4 no A7, 3 no A8, 6 no A9 e 5 no A10. Somando todas as aulas, 
temos: 4 + 5 + 4 + 3 + 6 + 5 = 27 aulas. Em 10 dias, estudando no máximo 2 aulas por dia, ele consegue assistir até
20 aulas (10 dias x 2 aulas/dia). Como 20 é menor que 27, ele não conseguirá terminar o conteúdo a tempo.  
Resposta: Não.

## Discussão

Few-shot é uma técnica de **formato**, não de raciocínio em si — ela funciona tanto para respostas diretas quanto para respostas com raciocínio, dependendo do que os exemplos ilustram. 

A vantagem é não precisar de nenhuma estrutura de grafo ou saída estruturada: só exemplos bem escolhidos no prompt. 

A desvantagem é que o raciocínio, quando presente, sai como texto livre dentro do `content` — não há garantia de formato, o que dificulta separar programaticamente "raciocínio" de "resposta final" de forma confiável. É exatamente esse problema que a próxima técnica resolve com saída estruturada.

# 3. Chain-of-Thought (CoT)

## 3.1 Conceito

Chain-of-Thought demanda ao modelo para gerar os **passos de raciocínio explicitamente**, antes de chegar à resposta final. Diferente do few-shot com raciocínio (que depende de exemplos), CoT costuma ser induzido diretamente por instrução (ex: "pense passo a passo") ou, de forma mais confiável para uso em aplicações, por **saída estruturada** com um campo dedicado ao raciocínio.

## 3.2 Saída estruturada com raciocínio separado

Vamos usar `with_structured_output` para garantir que o raciocínio e a resposta final venham em campos separados — isso facilita exibir cada um de forma consistente para o usuário.

In [59]:
class RespostaComRaciocinio(BaseModel):
    raciocinio: str = Field(description="Raciocínio passo a passo, numerado, explicando os cálculos e a lógica.")
    resposta_final: str = Field(description="Resposta final, direta e curta.")


cot_llm = llm.with_structured_output(RespostaComRaciocinio)

## 3.3 Aplicando Chain-of-Thought ao problema

In [60]:
resultado_cot = cot_llm.invoke([
    SystemMessage(content="""Resolva o problema pensando passo a passo, de forma explícita e numerada,
antes de dar a resposta final."""),
    HumanMessage(content=PROBLEMA_BRUNO),
])

print_reasoning(resultado_cot.raciocinio, resultado_cot.resposta_final)

Raciocínio:

1. Primeiro, vamos calcular o total de aulas restantes que Bruno precisa estudar. Ele já está no módulo A5, que tem
4 aulas, então ele precisa estudar os módulos A6, A7, A8, A9 e A10.  

2. O número de aulas em cada módulo é:  
   - A6: 5 aulas  
   - A7: 4 aulas  
   - A8: 3 aulas  
   - A9: 6 aulas  
   - A10: 5 aulas  

3. Agora, somamos todas as aulas restantes:  
   Total de aulas = 5 (A6) + 4 (A7) + 3 (A8) + 6 (A9) + 5 (A10) = 23 aulas  

4. Bruno pode estudar no máximo 2 aulas por dia.  
   Portanto, em 10 dias, ele pode estudar:  
   10 dias * 2 aulas/dia = 20 aulas  

5. Como ele precisa estudar 23 aulas e só consegue estudar 20 aulas em 10 dias, ele não conseguirá terminar o 
conteúdo a tempo.

Resposta final:

Não, Bruno não conseguirá terminar o conteúdo a tempo.

## Discussão

Com saída estruturada, o raciocínio e a resposta final vêm em campos previsíveis, o que permite exibir os dois de forma consistente (com `print_reasoning`), logar só o raciocínio para depuração, ou até esconder o raciocínio da interface do usuário final sem perder a garantia de que o modelo o produziu internamente antes de responder.

O custo de CoT, comparado à resposta direta, é meramente o tamanho da saída (mais tokens gerados) — ainda é uma única chamada ao modelo. É por isso que CoT costuma ser a técnica "padrão" quando se quer expor raciocínio sem multiplicar chamadas de LLM.

# 4. Self-Ask

## 4.1 Conceito

Self-Ask leva o Chain-of-Thought um passo além: em vez de um raciocínio em texto livre, o modelo **decompõe explicitamente a pergunta original em subperguntas**, respondendo cada uma antes de produzir a resposta final. A estrutura fica mais rígida e mais fácil de auditar pergunta por pergunta.

```text
Pergunta original: Bruno consegue terminar a tempo?

Subpergunta 1: Quantas aulas restam no total?
Resposta intermediária 1: 27 aulas.

Subpergunta 2: Quantos dias seriam necessários a 2 aulas por dia?
Resposta intermediária 2: 14 dias.

Subpergunta 3: 14 dias cabem nos 10 dias disponíveis?
Resposta intermediária 3: Não.

Resposta final: Não, Bruno não consegue terminar a tempo.
```

## 4.2 Saída estruturada de Self-Ask

In [62]:
class SubPerguntaRespondida(BaseModel):
    subpergunta: str = Field(description="Uma subpergunta necessária para responder a pergunta original.")
    resposta_intermediaria: str = Field(description="Resposta objetiva a essa subpergunta.")


class SelfAskResultado(BaseModel):
    subperguntas: List[SubPerguntaRespondida] = Field(
        description="Sequência de subperguntas e respostas necessárias para chegar à resposta final."
    )
    resposta_final: str = Field(description="Resposta final à pergunta original, com base nas subperguntas.")


self_ask_llm = llm.with_structured_output(SelfAskResultado)

## 4.3 Aplicando Self-Ask ao problema

In [63]:
resultado_self_ask = self_ask_llm.invoke([
    SystemMessage(content="""Decomponha a pergunta em subperguntas necessárias, na ordem em que
precisam ser respondidas, respondendo cada uma antes de seguir para a próxima.
Só então produza a resposta final."""),
    HumanMessage(content=PROBLEMA_BRUNO),
])

raciocinio_self_ask = "\n".join(
    f"Subpergunta: {sp.subpergunta}\nResposta intermediária: {sp.resposta_intermediaria}\n"
    for sp in resultado_self_ask.subperguntas
)

print_reasoning(raciocinio_self_ask, resultado_self_ask.resposta_final)

Raciocínio:

Subpergunta: Quantas aulas Bruno ainda precisa estudar nos módulos restantes?
Resposta intermediária: Bruno precisa estudar 4 (A6) + 4 (A7) + 3 (A8) + 6 (A9) + 5 (A10) = 22 aulas.

Subpergunta: Quantas aulas Bruno consegue estudar em 10 dias?
Resposta intermediária: Bruno consegue estudar 2 aulas por dia, então em 10 dias ele consegue estudar 2 * 10 = 20 
aulas.

Subpergunta: Bruno consegue terminar o conteúdo a tempo?
Resposta intermediária: Bruno precisa estudar 22 aulas, mas consegue estudar apenas 20 aulas em 10 dias.

Resposta final:

Não, Bruno não consegue terminar o conteúdo a tempo, pois ele precisa estudar 22 aulas, mas só conseguirá estudar 
20 aulas em 10 dias.

## Discussão

A vantagem do Self-Ask sobre o CoT genérico é a **estrutura**: cada subpergunta e resposta intermediária é um item claramente delimitado, o que facilita apontar exatamente onde um raciocínio errado aconteceu (por exemplo, se a subpergunta 2 estiver certa mas a 3 estiver errada). Isso também abre a porta para **verificar ou substituir** uma subpergunta específica por uma tool real (por exemplo, uma subpergunta poderia ser respondida por uma chamada de calculadora em vez de o modelo calcular de cabeça) — o que aproxima Self-Ask de uma versão mais estruturada de ReAct.

# 5. Self-Consistency

## 5.1 Conceito

Self-Consistency parte de uma premissa: pedirmos a mesma pergunta várias vezes com Chain-of-Thought e alguma aleatoriedade (temperatura > 0), o modelo pode gerar **caminhos de raciocínio diferentes**. Esses caminhos diferentes às vezes chegam a respostas diferentes. Isso visa tratar o aspecto de confiar em uma única cadeia de raciocínio.

1. Gera **N cadeias de raciocínio independentes** para a mesma pergunta;
2. Extrai a resposta final de cada uma;
3. Faz uma **votação por maioria** entre as respostas finais.

```text
Cadeia 1 → resposta: "Não"
Cadeia 2 → resposta: "Não"
Cadeia 3 → resposta: "Sim"
                          ↓
                 votação majoritária: "Não"
```

Como pedimos para exibir todo raciocínio explícito ao usuário, vamos mostrar **todas as cadeias geradas**, não só a resposta vencedora.

## 5.2 Gerando múltiplas cadeias de raciocínio

Usamos um modelo com temperatura mais alta, para que as N chamadas produzam caminhos de raciocínio diferentes entre si.

In [64]:
llm_com_temperatura = init_chat_model("openai:gpt-4o-mini", temperature=0.9)
cot_llm_temperatura = llm_com_temperatura.with_structured_output(RespostaComRaciocinio)


def self_consistency(pergunta: str, n_amostras: int = 5) -> dict:
    cadeias = []

    for _ in range(n_amostras):
        resultado = cot_llm_temperatura.invoke([
            SystemMessage(content="Resolva o problema pensando passo a passo antes de responder."),
            HumanMessage(content=pergunta),
        ])
        cadeias.append(resultado)

    respostas = [c.resposta_final.strip().lower() for c in cadeias]
    contagem = Counter(respostas)
    resposta_majoritaria, votos = contagem.most_common(1)[0]

    return {
        "cadeias": cadeias,
        "contagem": contagem,
        "resposta_majoritaria": resposta_majoritaria,
        "votos": votos,
        "total_amostras": n_amostras,
    }

## 5.3 Executando Self-Consistency e exibindo todas as cadeias

In [65]:
resultado_sc = self_consistency(PROBLEMA_BRUNO, n_amostras=5)

for i, cadeia in enumerate(resultado_sc["cadeias"]):
    print(f"\n[bold]--- Cadeia {i + 1} ---[/bold]")
    print_reasoning(cadeia.raciocinio, cadeia.resposta_final)

print("\n[bold]Contagem de votos:[/bold]")
print_json(dict(resultado_sc["contagem"]))

print(f"\n[bold]Resposta por maioria ({resultado_sc['votos']}/{resultado_sc['total_amostras']} votos):[/bold] {resultado_sc['resposta_majoritaria']}")

--- Cadeia 1 ---

Raciocínio:

1. Vamos calcular o total de aulas que Bruno precisa estudar nos módulos restantes. Ele já está no módulo A5, então
ele precisa estudar os módulos A6, A7, A8, A9 e A10.  

2. O cálculo das aulas de cada módulo é o seguinte:  
   - A5: 4 aulas (já iniciado)  
   - A6: 5 aulas  
   - A7: 4 aulas  
   - A8: 3 aulas  
   - A9: 6 aulas  
   - A10: 5 aulas  

3. Somando as aulas dos módulos restantes:  
   A6 + A7 + A8 + A9 + A10 = 5 + 4 + 3 + 6 + 5 = 23 aulas.  

4. Bruno pode estudar, no máximo, 2 aulas por dia.  

5. Calculando quantos dias são necessários para estudar as 23 aulas:  
   Total de aulas: 23  
   Aulas por dia: 2  
   Dias necessários = 23 / 2 = 11,5 dias.  

6. Como Bruno não pode estudar por meio dia, ele precisaria de 12 dias para completar todo o conteúdo.  

7. Como ele tem apenas 10 dias antes da prova final, ele não conseguirá terminar o conteúdo a tempo.

Resposta final:

Não, Bruno não conseguirá terminar o conteúdo a tempo.

...

Resposta final:

Não, Bruno não conseguirá terminar o conteúdo a tempo.

Contagem de votos:

{
  "não, bruno não conseguirá terminar o conteúdo a tempo.": 2,
  "não, bruno não consegue terminar o conteúdo a tempo.": 3
}

Resposta por maioria (3/5 votos): não, bruno não consegue terminar o conteúdo a tempo.

## Discussão

Self-Consistency troca **custo por confiabilidade**: em vez de confiar em uma única cadeia de raciocínio (que pode ter um erro de conta pontual), várias cadeias independentes tendem a "cancelar" erros aleatórios, desde que a maioria delas raciocine corretamente. É particularmente útil em problemas com contas ou lógica onde um único deslize muda a resposta.

O custo é literal: `n_amostras` vezes o custo de uma única chamada com CoT. Por isso, essa técnica costuma ser reservada para perguntas de alto risco (ex: decisões importantes) e não para todo turno de uma conversa.

# 6. ReAct

ReAct é, por construção, uma técnica de **reasoning explícito**: cada turno do agente intercala uma mensagem de raciocínio (ou decisão de chamar uma tool) com uma observação real. Ou seja, todo o histórico de mensagens *é* o raciocínio exposto — não precisamos de nenhum campo estruturado extra para isso.

Vamos aplicar o mesmo problema do Bruno, agora dando ao agente uma tool de cálculo, para ilustrar como o raciocínio de ReAct aparece com ações reais.

In [68]:
@tool_decorator
def calcular_dias_necessarios(total_aulas: int, aulas_por_dia: int) -> str:
    """Calcula quantos dias são necessários para assistir a um total de aulas, dado um limite de aulas por dia."""
    import math
    dias = math.ceil(total_aulas / aulas_por_dia)
    return json.dumps({"total_aulas": total_aulas, "aulas_por_dia": aulas_por_dia, "dias_necessarios": dias})


llm_com_calculadora = llm.bind_tools([calcular_dias_necessarios])


def loop_react_com_raciocinio(pergunta: str, max_turnos: int = 4) -> list[BaseMessage]:
    messages: list[BaseMessage] = [
        SystemMessage(content="""Resolva o problema usando a tool de cálculo quando precisar somar dias.
Explique seu raciocínio em texto antes de decidir usar a tool."""),
        HumanMessage(content=pergunta),
    ]

    for _ in range(max_turnos):
        response = llm_com_calculadora.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            observacao = calcular_dias_necessarios.invoke(tool_call["args"])
            messages.append(ToolMessage(content=observacao, tool_call_id=tool_call["id"]))

    return messages


mensagens_react = loop_react_com_raciocinio(PROBLEMA_BRUNO)

for i, m in enumerate(mensagens_react):
    print(f"\n[bold][{i}] {m.__class__.__name__}[/bold]")
    if isinstance(m, AIMessage) and m.tool_calls:
        print("tool_calls:", [(tc["name"], tc["args"]) for tc in m.tool_calls])
        if m.content:
            print(m.content)
    else:
        print(m.content)

[0] SystemMessage

Resolva o problema usando a tool de cálculo quando precisar somar dias.
Explique seu raciocínio em texto antes de decidir usar a tool.

[1] HumanMessage

Bruno está no módulo A5. Os módulos restantes do curso são:
A5 (4 aulas), A6 (5 aulas), A7 (4 aulas), A8 (3 aulas), A9 (6 aulas), A10 (5 aulas).
Cada aula dura 50 minutos. Bruno tem 10 dias antes da prova final e consegue
estudar, no máximo, 2 aulas por dia. Ele consegue terminar o conteúdo a tempo?

[2] AIMessage

...

Para determinar se Bruno consegue terminar o conteúdo a tempo, precisamos primeiro calcular o total de aulas 
restantes que ele precisa assistir e, em seguida, calcular quantos dias ele precisará para assistir a essas aulas, 
considerando que ele pode assistir a no máximo 2 aulas por dia.

1. **Total de aulas restantes**:
   - A5: 4 aulas
   - A6: 5 aulas
   - A7: 4 aulas
   - A8: 3 aulas
   - A9: 6 aulas
   - A10: 5 aulas

   Somando todas as aulas:
   [
   4 + 5 + 4 + 3 + 6 + 5 = 27 \text{ aulas}
   \]

2. **Cálculo dos dias necessários**:
   Bruno pode assistir a 2 aulas por dia. Portanto, precisamos calcular quantos dias ele precisará para assistir a 
27 aulas.

Agora, vamos usar a ferramenta de cálculo para determinar quantos dias são necessários para assistir a 27 aulas, 
com um limite de 2 aulas por dia.

[3] ToolMessage

{"total_aulas": 27, "aulas_por_dia": 2, "dias_necessarios": 14}

[4] AIMessage

Bruno precisaria de 14 dias para assistir a todas as 27 aulas, considerando que ele pode assistir a no máximo 2 
aulas por dia. Como ele tem apenas 10 dias antes da prova final, ele não conseguirá terminar o conteúdo a tempo.

## Discussão

Note que, aqui, **não precisamos de um `print_reasoning` separado** — o raciocínio já está exposto naturalmente como parte da sequência de mensagens que imprimimos. Essa é uma diferença estrutural importante em relação a CoT, Self-Ask e Self-Consistency: nessas três, o raciocínio é um *artefato adicional* que produzimos deliberadamente (um campo, uma lista de subperguntas, várias cadeias); no ReAct, o raciocínio é *inerente* ao próprio formato de execução do agente.

Para a implementação completa em LangGraph, com estado, nós e roteamento condicional, veja o notebook de Planning (seção 1).

# 7. Least-to-Most

## 7.1 Conceito

Least-to-Most decompõe o problema em subproblemas ordenados **do mais simples para o mais complexo**, resolvendo cada um em sequência e usando as respostas anteriores como contexto para o próximo. A diferença central para o Self-Ask é a **ordem deliberada por dificuldade crescente**: cada subproblema deve depender apenas de subproblemas já resolvidos, nunca de subproblemas futuros.

```text
Subproblema 1 (mais simples): Quantas aulas restam no total?
  → 27 aulas.

Subproblema 2 (usa o resultado anterior): Quantos dias são necessários a 2 aulas/dia?
  → 14 dias.

Subproblema 3 (mais complexo, decide a resposta): 14 dias cabem em 10 dias disponíveis?
  → Não.
```

Isso é particularmente útil quando o problema tem uma estrutura de **dependência clara** entre subpartes — resolver a parte simples primeiro reduz o espaço de erro da parte mais complexa.

## 7.2 Saída estruturada da decomposição

In [69]:
class Decomposicao(BaseModel):
    subproblemas: List[str] = Field(
        description="Lista ordenada de subproblemas, do mais simples ao mais complexo, cada um resolvível com base nos anteriores."
    )


decompositor = llm.with_structured_output(Decomposicao)

## 7.3 Resolvendo cada subproblema em ordem

In [70]:
def least_to_most(pergunta: str) -> dict:
    decomposicao = decompositor.invoke([
        SystemMessage(content="""Decomponha o problema em no máximo 4 subproblemas, do mais simples ao
mais complexo, de forma que cada subproblema possa ser resolvido usando os resultados dos anteriores."""),
        HumanMessage(content=pergunta),
    ])

    respostas_intermediarias = []

    for subproblema in decomposicao.subproblemas:
        contexto = "\n".join(
            f"- {sp}: {r}" for sp, r in zip(decomposicao.subproblemas, respostas_intermediarias)
        ) or "Nenhum subproblema resolvido ainda."

        resposta = llm.invoke([
            SystemMessage(content="Resolva apenas o subproblema atual, de forma objetiva, usando o contexto disponível."),
            HumanMessage(content=f"Problema original: {pergunta}\n\nContexto:\n{contexto}\n\nSubproblema atual: {subproblema}"),
        ])

        respostas_intermediarias.append(resposta.content)

    resposta_final = llm.invoke([
        SystemMessage(content="Sintetize a resposta final com base nos subproblemas resolvidos."),
        HumanMessage(content=(
            f"Problema original: {pergunta}\n\n"
            + "\n".join(f"- {sp}: {r}" for sp, r in zip(decomposicao.subproblemas, respostas_intermediarias))
        )),
    ])

    return {
        "subproblemas": decomposicao.subproblemas,
        "respostas_intermediarias": respostas_intermediarias,
        "resposta_final": resposta_final.content,
    }

## 7.4 Executando Least-to-Most

In [71]:
resultado_ltm = least_to_most(PROBLEMA_BRUNO)

raciocinio_ltm = "\n".join(
    f"Subproblema: {sp}\nResposta: {r}\n"
    for sp, r in zip(resultado_ltm["subproblemas"], resultado_ltm["respostas_intermediarias"])
)

print_reasoning(raciocinio_ltm, resultado_ltm["resposta_final"])

Raciocínio:

Subproblema: Calcular o total de aulas restantes que Bruno precisa estudar.
Resposta: Bruno está no módulo A5 e ainda precisa estudar os seguintes módulos:

- A6: 5 aulas
- A7: 4 aulas
- A8: 3 aulas
- A9: 6 aulas
- A10: 5 aulas

Total de aulas restantes:

A6 + A7 + A8 + A9 + A10 = 5 + 4 + 3 + 6 + 5 = 23 aulas

Portanto, Bruno precisa estudar um total de 23 aulas restantes.

Subproblema: Calcular o total de aulas que Bruno pode estudar em 10 dias.
Resposta: Bruno pode estudar, no máximo, 2 aulas por dia. Em 10 dias, ele pode estudar:

2 aulas/dia * 10 dias = 20 aulas

Portanto, Bruno pode estudar um total de 20 aulas em 10 dias.

Subproblema: Comparar o total de aulas restantes com o total de aulas que Bruno pode estudar para determinar se ele
consegue terminar a tempo.
Resposta: Bruno precisa estudar um total de 23 aulas restantes, mas ele pode estudar apenas 20 aulas em 10 dias. 

Comparando os dois valores:
- Aulas restantes: 23
- Aulas que pode estudar: 20

Como 23 > 20, Bruno não conseguirá terminar o conteúdo a tempo.

Resposta final:

Bruno precisa estudar um total de 23 aulas restantes, mas pode estudar apenas 20 aulas em 10 dias. Como 23 é maior 
que 20, ele não conseguirá terminar o conteúdo a tempo antes da prova final.

## Discussão

Least-to-Most é similar ao Self-Ask na superfície (ambos decompõem em subperguntas/subproblemas), mas a ênfase é diferente: Self-Ask decompõe **o que precisa ser perguntado**; Least-to-Most decompõe **por ordem de dificuldade**, garantindo que cada passo só dependa de passos anteriores já resolvidos. Isso tende a ajudar mais em problemas com estrutura de dependência clara (matemática, lógica em cadeia) do que em perguntas abertas onde a ordem das subperguntas é menos óbvia.

O custo é maior que CoT simples: uma chamada para decompor, mais uma chamada por subproblema, mais uma chamada final de síntese.

# 8. Tree of Thoughts

Em vez de uma única cadeia de raciocínio, o modelo gera **vários candidatos de raciocínio ou solução em paralelo**, avalia cada um, e segue com o melhor.

Aplicado ao problema do Bruno, isso significaria gerar múltiplas formas de calcular ou verificar a resposta (por exemplo, uma abordagem por total de minutos, outra por total de aulas, outra por dias necessários), avaliar qual é mais confiável, e reportar a que teve melhor avaliação — de forma parecida com o exemplo de estratégias de revisão do notebook anterior.

Assim como no Self-Consistency, o raciocínio explícito aqui não é um único texto — são **múltiplos candidatos, cada um com sua própria justificativa**, mais a avaliação que decide entre eles. A diferença central para Self-Consistency é que, no ToT, os candidatos podem ser **abordagens diferentes** (não apenas repetições da mesma abordagem com aleatoriedade), e a seleção usa um critério de avaliação explícito, não apenas votação por maioria.

Para a implementação completa em LangGraph (geração de candidatos + avaliação + seleção), veja o notebook de Planning (seção 6).

# 9. Comparando as sete técnicas

| Técnica | Raciocínio explícito? | Nº aprox. de chamadas de LLM | Quando ajuda mais |
|---|---|---|---|
| Few-Shot | Só se os exemplos incluírem raciocínio | 1 | Orientar formato/estilo de resposta |
| Chain-of-Thought | Sim, em texto ou campo estruturado | 1 | Problemas que exigem alguns passos lógicos |
| Self-Ask | Sim, como subperguntas e respostas | 1 (mas maior, com toda a decomposição) | Perguntas que se decompõem naturalmente em subperguntas |
| Self-Consistency | Sim, múltiplas cadeias completas | N (uma por amostra) | Problemas de risco alto, onde vale gastar mais para confiar mais |
| ReAct | Sim, inerente ao histórico de mensagens | 1 por turno | Tarefas que precisam intercalar raciocínio com ações reais |
| Least-to-Most | Sim, por subproblema em ordem de dificuldade | 1 (decompor) + 1 por subproblema + 1 (síntese) | Problemas com dependência clara entre partes |
| Tree of Thoughts | Sim, por candidato avaliado | 2+ (gerar N candidatos + avaliar) | Múltiplas abordagens possíveis, escolher errado é caro |

Um ponto que vale destacar: **exibir raciocínio para o usuário não é grátis, mas normalmente também não é caro** — na maioria das técnicas (CoT, Self-Ask, Least-to-Most, ReAct), o raciocínio já é gerado de qualquer forma; a única mudança é exibi-lo em vez de escondê-lo. As exceções são Self-Consistency e Tree of Thoughts, que **multiplicam** chamadas de LLM independentemente de exibir ou não o raciocínio — nesses casos, exibir todas as cadeias/candidatos é, na prática, gratuito adicional sobre um custo que já existia.

# 10. Boas práticas

## 10.1 Prefira saída estruturada a parsing de texto livre

Sempre que possível, separe raciocínio de resposta final em campos distintos (`with_structured_output`), em vez de tentar extrair isso de um texto livre com regex — é mais robusto e mais fácil de exibir de forma consistente.

## 10.2 Decida deliberadamente o que mostrar ao usuário

Nem todo raciocínio exposto precisa aparecer integralmente na interface final. É comum logar o raciocínio completo para depuração, mas mostrar ao usuário só um resumo, ou uma versão colapsável.

## 10.3 Escolha a técnica pelo formato do problema, não pela moda

Perguntas simples não precisam de Self-Consistency; problemas de alto risco raramente devem confiar em uma única cadeia de CoT sem verificação.

## 10.4 Combine técnicas quando fizer sentido

Nada impede usar Least-to-Most para decompor o problema e Self-Consistency dentro de um subproblema específico de alto risco — as técnicas não são mutuamente exclusivas.

## 10.5 Cuidado com o custo de Self-Consistency e Tree of Thoughts

Ambas multiplicam chamadas de LLM. Defina sempre um número máximo de amostras/candidatos e avalie se o ganho de confiabilidade justifica o custo adicional.

# 11. Exercícios de fixação

## Exercício 1 — Few-shot com exemplos ruins

Reescreva `EXEMPLOS_FEW_SHOT_COM_RACIOCINIO` incluindo um exemplo com raciocínio **incorreto**. Observe se isso piora a resposta do modelo para o problema do Bruno.

In [42]:
# Escreva sua solução aqui.

## Exercício 2 — CoT com verificação

Depois de obter `resultado_cot`, adicione uma segunda chamada ao modelo pedindo para **verificar** o raciocínio produzido (procurar erros de conta) antes de aceitar a resposta final.

In [43]:
# Escreva sua solução aqui.

## Exercício 3 — Self-Ask com subpergunta errada

Force manualmente uma resposta intermediária errada em uma das subperguntas de `SelfAskResultado` e observe como isso afeta a resposta final gerada a partir dela.

In [44]:
# Escreva sua solução aqui.

## Exercício 4 — Self-Consistency com poucas amostras

Rode `self_consistency` com `n_amostras=2` e depois com `n_amostras=9`. Compare a confiança da votação majoritária (quantos votos o vencedor recebeu) entre as duas execuções.

In [45]:
# Escreva sua solução aqui.

## Exercício 5 — ReAct com raciocínio insuficiente

Modifique o system prompt de `loop_react_com_raciocinio` para **não** pedir explicação em texto antes da tool call. Compare a qualidade do raciocínio exposto nas mensagens.

In [46]:
# Escreva sua solução aqui.

## Exercício 6 — Least-to-Most com subproblema fora de ordem

Force manualmente uma ordem diferente em `decomposicao.subproblemas` (por exemplo, colocando o subproblema mais complexo primeiro) e observe se isso prejudica as respostas intermediárias.

In [47]:
# Escreva sua solução aqui.

## Exercício 7 — Escolhendo a técnica certa

Para cada uma das situações abaixo, indique qual das sete técnicas você usaria e justifique:

1. Uma pergunta simples de fato sobre a carga horária de um módulo.
2. Um problema de matemática com várias etapas dependentes entre si.
3. Uma decisão de alto risco, onde errar a resposta tem consequência real para o aluno.
4. Uma tarefa que precisa consultar dados reais (tools) para responder corretamente.

In [48]:
# Escreva sua resposta aqui.

# 12. Resumo da aula

Neste notebook, vimos sete técnicas de reasoning:

- **Few-Shot**: orienta por exemplos; expõe raciocínio só se os exemplos o incluírem.
- **Chain-of-Thought**: pede raciocínio passo a passo, idealmente em campo estruturado separado da resposta.
- **Self-Ask**: decompõe a pergunta em subperguntas explícitas, respondidas em sequência.
- **Self-Consistency**: gera várias cadeias de raciocínio independentes e vota pela resposta majoritária.
- **ReAct**: intercala raciocínio e ação; o raciocínio é inerente ao histórico de mensagens.
- **Least-to-Most**: decompõe do subproblema mais simples ao mais complexo, cada um usando os anteriores.
- **Tree of Thoughts**: gera múltiplos candidatos de raciocínio, avalia e segue com o melhor.

Um ponto central da aula: sempre que a técnica produz raciocínio explícito, vale a pena **expor esse raciocínio ao usuário** — na maioria dos casos (CoT, Self-Ask, Least-to-Most, ReAct), isso não tem custo adicional, porque o raciocínio já foi gerado de qualquer forma.

## 12.1 Checklist de compreensão

Antes de avançar, verifique se você consegue responder:

1. Qual a diferença entre reasoning implícito e explícito?
2. Por que few-shot, por si só, não garante raciocínio exposto?
3. Por que saída estruturada é preferível a parsing de texto livre para separar raciocínio de resposta final?
4. O que diferencia Self-Ask de Least-to-Most, já que ambos decompõem em subpartes?
5. Por que o raciocínio do ReAct não precisa de um campo estruturado extra?
6. Como Self-Consistency usa múltiplas cadeias para aumentar confiabilidade?
7. Em que Tree of Thoughts se parece e se diferencia de Self-Consistency?
8. Quais técnicas multiplicam o número de chamadas de LLM, e por quê?

## 12.2 Próximos passos

A partir daqui, você pode evoluir os exemplos deste notebook para incluir:

- combinar Least-to-Most com Self-Consistency no subproblema mais arriscado;
- usar Self-Ask com subperguntas respondidas por tools reais, em vez de pelo próprio modelo;
- registrar métricas de quantas vezes o raciocínio exposto continha um erro identificável;
- interfaces de usuário com o raciocínio colapsável (visível sob demanda);
- combinar as técnicas de reasoning desta aula com as estratégias de planejamento do notebook anterior.

# 13. Referências

- Brown et al. — Language Models are Few-Shot Learners (GPT-3)
- Wei et al. — Chain-of-Thought Prompting Elicits Reasoning in Large Language Models
- Press et al. — Measuring and Narrowing the Compositionality Gap in Language Models (Self-Ask)
- Wang et al. — Self-Consistency Improves Chain of Thought Reasoning in Language Models
- Yao et al. — ReAct: Synergizing Reasoning and Acting in Language Models
- Zhou et al. — Least-to-Most Prompting Enables Complex Reasoning in Large Language Models
- Yao et al. — Tree of Thoughts: Deliberate Problem Solving with Large Language Models
- Notebooks anteriores da sequência: N3 (LangGraph com LLMs), N4 (Ferramentas), N8 (Planning)